# 01. Factor Exploration

Research sandbox for point-in-time factor experiments using `data/research.db`.

Goals:
- Load KOSPI200 price and DART financial data safely into pandas.
- Build point-in-time factor snapshots with `disclosed_at <= as_of`.
- Try value equations, regression features, and custom magic-formula ideas.

In [ ]:
from __future__ import annotations

import sqlite3
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "data" / "research.db").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_PATH = PROJECT_ROOT / "data" / "research.db"
assert DB_PATH.exists(), f"research.db not found: {DB_PATH}"
DB_PATH

## Database helpers

The helpers below keep queries explicit and point-in-time safe. Financial data must always be filtered by `disclosed_at <= as_of`.

In [ ]:
def connect() -> sqlite3.Connection:
    return sqlite3.connect(DB_PATH)


def read_sql(sql: str, params: tuple = ()) -> pd.DataFrame:
    with connect() as conn:
        return pd.read_sql_query(sql, conn, params=params)


def load_stock_master(market: str | None = "KOSPI") -> pd.DataFrame:
    where = "WHERE market = ?" if market else ""
    params = (market,) if market else ()
    df = read_sql(f"""
        SELECT code, name, market, sector, industry, listed_at, delisted_at, is_delisted
        FROM stocks
        {where}
        ORDER BY code
    """, params)
    for col in ["listed_at", "delisted_at"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def load_prices(start: str | None = None, end: str | None = None, codes: list[str] | None = None) -> pd.DataFrame:
    clauses = []
    params: list[object] = []
    if start:
        clauses.append("date >= ?")
        params.append(start)
    if end:
        clauses.append("date <= ?")
        params.append(end)
    if codes:
        placeholders = ",".join("?" for _ in codes)
        clauses.append(f"stock_code IN ({placeholders})")
        params.extend([str(code).zfill(6) for code in codes])
    where = "WHERE " + " AND ".join(clauses) if clauses else ""
    df = read_sql(f"""
        SELECT stock_code, date, open, high, low, close, volume, COALESCE(adj_close, close) AS adj_close
        FROM prices_daily
        {where}
        ORDER BY date, stock_code
    """, tuple(params))
    df["date"] = pd.to_datetime(df["date"])
    numeric_cols = ["open", "high", "low", "close", "volume", "adj_close"]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    return df


def load_financials(codes: list[str] | None = None, disclosed_asof: str | None = None) -> pd.DataFrame:
    clauses = []
    params: list[object] = []
    if codes:
        placeholders = ",".join("?" for _ in codes)
        clauses.append(f"stock_code IN ({placeholders})")
        params.extend([str(code).zfill(6) for code in codes])
    if disclosed_asof:
        clauses.append("disclosed_at <= ?")
        params.append(disclosed_asof)
    where = "WHERE " + " AND ".join(clauses) if clauses else ""
    df = read_sql(f"""
        SELECT stock_code, fiscal_period, disclosed_at, revenue, operating_income,
               net_income, total_assets, total_equity, eps, bps
        FROM financials
        {where}
        ORDER BY stock_code, disclosed_at, fiscal_period
    """, tuple(params))
    for col in ["fiscal_period", "disclosed_at"]:
        df[col] = pd.to_datetime(df[col])
    numeric_cols = ["revenue", "operating_income", "net_income", "total_assets", "total_equity", "eps", "bps"]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    return df

## Point-in-time factor snapshot

`factor_snapshot(as_of)` selects the latest price and latest disclosed financial row available as of that date.

In [ ]:
def factor_snapshot(as_of: str, market: str = "KOSPI") -> pd.DataFrame:
    stocks = load_stock_master(market=market)
    as_of_ts = pd.Timestamp(as_of)
    alive = stocks[
        (stocks["listed_at"] <= as_of_ts)
        & (stocks["delisted_at"].isna() | (stocks["delisted_at"] > as_of_ts))
    ].copy()
    codes = alive["code"].tolist()

    prices = load_prices(end=as_of, codes=codes)
    latest_price = (
        prices.sort_values(["stock_code", "date"])
        .groupby("stock_code", as_index=False)
        .tail(1)[["stock_code", "date", "adj_close", "volume"]]
        .rename(columns={"date": "price_date", "adj_close": "price"})
    )

    financials = load_financials(codes=codes, disclosed_asof=as_of)
    latest_fin = (
        financials.sort_values(["stock_code", "disclosed_at", "fiscal_period"])
        .groupby("stock_code", as_index=False)
        .tail(1)
    )

    df = alive.merge(latest_price, left_on="code", right_on="stock_code", how="left")
    df = df.merge(latest_fin, left_on="code", right_on="stock_code", how="left", suffixes=("", "_fin"))
    df["per"] = df["price"] / df["eps"]
    df["pbr"] = df["price"] / df["bps"]
    df["roe"] = df["net_income"] / df["total_equity"]
    df.loc[df["eps"] <= 0, "per"] = np.nan
    df.loc[df["bps"] <= 0, "pbr"] = np.nan
    df.loc[df["total_equity"] <= 0, "roe"] = np.nan
    df["market_cap_proxy"] = df["price"] * df["volume"]
    return df


snapshot = factor_snapshot("2025-12-31")
snapshot[["code", "name", "price", "eps", "bps", "per", "pbr", "roe", "disclosed_at"]].head()

## Clean features and try a simple equation

This is a sandbox starting point. Replace `score` with whatever equation you are studying.

In [ ]:
def zscore(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    std = values.std(ddof=0)
    if pd.isna(std) or std == 0:
        return values * 0
    return (values - values.mean()) / std


features = snapshot.copy()
features = features.replace([np.inf, -np.inf], np.nan)
features = features.dropna(subset=["per", "pbr", "roe", "price"])
features["score"] = -zscore(features["per"]) - 0.8 * zscore(features["pbr"]) + zscore(features["roe"])
features.sort_values("score", ascending=False)[["code", "name", "per", "pbr", "roe", "score"]].head(20)

## Regression sketch

The next cell builds a toy next-period return target from a single date. For serious research, extend this into a monthly/quarterly panel.

In [ ]:
def add_forward_return(df: pd.DataFrame, as_of: str, horizon_days: int = 63) -> pd.DataFrame:
    as_of_ts = pd.Timestamp(as_of)
    end_ts = as_of_ts + pd.Timedelta(days=horizon_days * 2)
    px = load_prices(start=as_of, end=end_ts.date().isoformat(), codes=df["code"].tolist())
    start_px = px[px["date"] >= as_of_ts].sort_values(["stock_code", "date"]).groupby("stock_code").head(1)
    future_px = px[px["date"] > as_of_ts].sort_values(["stock_code", "date"]).groupby("stock_code").nth(min(horizon_days, max(len(px), 1) - 1)).reset_index()
    returns = start_px[["stock_code", "adj_close"]].rename(columns={"adj_close": "start_price"})
    returns = returns.merge(future_px[["stock_code", "adj_close"]].rename(columns={"adj_close": "future_price"}), on="stock_code", how="inner")
    returns["forward_return"] = returns["future_price"] / returns["start_price"] - 1
    return df.merge(returns[["stock_code", "forward_return"]], left_on="code", right_on="stock_code", how="left")


model_df = add_forward_return(features, "2025-12-31").dropna(subset=["per", "pbr", "roe", "forward_return"])
X = model_df[["per", "pbr", "roe"]].copy()
X[["per", "pbr", "roe"]] = StandardScaler().fit_transform(X)
y = model_df["forward_return"]

if len(model_df) >= 10:
    model = Ridge(alpha=1.0).fit(X, y)
    pred = model.predict(X)
    print("R2", r2_score(y, pred))
    display(pd.Series(model.coef_, index=X.columns, name="coef"))
else:
    print("Not enough rows for regression sample.")